In [31]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
from IPython.display import display

In [34]:
import pandas as pd

df = pd.read_csv("../data/signals_markings_signs/signals_signs_markings_combined.csv")

/var/folders/0q/f6wdygs95dz97f63ll8xv7mc0000gn/T/ipykernel_31087/3037373250.py:3: DtypeWarning: Columns (3,7,8,9,10,11,14,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/signals_markings_signs/signals_signs_markings_combined.csv")


In [35]:
import datamart_profiler

metadata = datamart_profiler.process_dataset(df, include_sample=True, plots=True)

In [36]:
import json

with open("../data_profiles/signals_markings_signs.json", "w") as f:
    json.dump(metadata, f, indent=2)

In [24]:
print("\nAvailable columns:", df.columns.tolist())

# View sample rows from APS
target = df[df["source"] == "accessible_ped_signal"].copy()
target["POINT_X"] = pd.to_numeric(target["POINT_X"], errors="coerce")
target["POINT_Y"] = pd.to_numeric(target["POINT_Y"], errors="coerce")
target = target.dropna(subset=["POINT_X", "POINT_Y"])
print("Valid APS points:", len(target))


Available columns: ['POINT_X', 'POINT_Y', 'source', 'borough', 'sign_x_coord', 'sign_y_coord', 'Unique Key', 'Created Date', 'Closed Date', 'Status', 'Complaint Type', 'Descriptor', 'Latitude', 'Longitude', 'Street Name', 'Incident Zip', 'Borough']
Valid APS points: 3343


In [25]:
# Create GeoDataFrame (assume coordinates are EPSG:4326)
gdf = gpd.GeoDataFrame(
    target,
    geometry=gpd.points_from_xy(target["POINT_X"], target["POINT_Y"]),
    crs="EPSG:4326"
)

# Create folium map
m = folium.Map(location=[40.7128, -74.0060], zoom_start=12, tiles="CartoDB positron")
marker_cluster = MarkerCluster().add_to(m)

# Add APS markers to the map
for _, row in gdf.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=f"APS Location"
    ).add_to(marker_cluster)

# Show the map
display(m)


In [30]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
from IPython.display import display

# ------------------------------------------------------------------
# 1. Load the combined file (adjust the path if you store it elsewhere)
# ------------------------------------------------------------------
csv_path = "../data/signals_markings_signs/signals_signs_markings_combined.csv"
df = pd.read_csv(csv_path, low_memory=False)

# ------------------------------------------------------------------
# 2. Build one GeoDataFrame in EPSG:4326 for **every** source
#    • APS  ➜ POINT_X / POINT_Y   (NY State-Plane 2263)
#    • Street-sign work orders ➜ sign_x_coord / sign_y_coord (2263)
#    • Traffic signals         ➜ Latitude / Longitude        (already 4326)
# ------------------------------------------------------------------
layers = []

# Accessible pedestrian signals (APS)
aps = df.loc[df["source"] == "accessible_ped_signal"].dropna(subset=["POINT_X", "POINT_Y"])
if not aps.empty:
    gdf_aps = gpd.GeoDataFrame(
        aps,
        geometry=gpd.points_from_xy(aps["POINT_X"], aps["POINT_Y"]),
        crs="EPSG:2263"          # NY-State-Plane feet
    ).to_crs("EPSG:4326")
    layers.append(gdf_aps[["source", "geometry"]])

# Street-sign work orders
ss = df.loc[df["source"] == "street_sign"].dropna(subset=["sign_x_coord", "sign_y_coord"])
if not ss.empty:
    gdf_ss = gpd.GeoDataFrame(
        ss,
        geometry=gpd.points_from_xy(ss["sign_x_coord"], ss["sign_y_coord"]),
        crs="EPSG:2263"
    ).to_crs("EPSG:4326")
    layers.append(gdf_ss[["source", "geometry"]])

# Traffic signals (new 311 feed ➜ already lat/lon)
ts = df.loc[df["source"] == "traffic_signal"].dropna(subset=["Latitude", "Longitude"])
if not ts.empty:
    gdf_ts = gpd.GeoDataFrame(
        ts,
        geometry=gpd.points_from_xy(ts["Longitude"], ts["Latitude"]),
        crs="EPSG:4326"
    )
    layers.append(gdf_ts[["source", "geometry"]])

# ------------------------------------------------------------------
# 3. Concatenate & take a random sample (up to 1 000 rows)
# ------------------------------------------------------------------
all_points = gpd.GeoDataFrame(pd.concat(layers, ignore_index=True), crs="EPSG:4326")
gdf_sample = all_points.sample(n=min(1_000, len(all_points)), random_state=42)

# ------------------------------------------------------------------
# 4. Make the map
# ------------------------------------------------------------------
m = folium.Map(location=[40.7128, -74.0060], zoom_start=11, tiles="CartoDB positron")
marker_cluster = MarkerCluster().add_to(m)

for _, row in gdf_sample.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        popup=f"Source: {row['source']}",
        fill=True,
        fill_opacity=0.8
    ).add_to(marker_cluster)

display(m)